In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "catcher-feedback-eval"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 1. 단건 테스트 — generate_monthly_feedback()

분석 월(YYYY-MM)의 전체 소비를 분석해 월간 피드백을 생성합니다.

In [ ]:
from catcher_llm.services.consumption_feedback.monthly_feedback import generate_monthly_feedback

result = generate_monthly_feedback(
    member_id=1,
    analysis_month="2024-01"
)

if result.error:
    print("오류:", result.error)
else:
    print("[월간 피드백]")
    print(result.feedback.feedback_message)

# 2. target 함수 정의

In [ ]:
def target(inputs: dict):
    result = generate_monthly_feedback(
        member_id=inputs["member_id"],
        analysis_month=inputs["analysis_month"]
    )

    if result.error:
        return {"answer": f"[ERROR] {result.error}", "month_total": 0, "top_category": ""}

    month_total = (
        result.monthly_analysis.stable_metrics.month_total
        if result.monthly_analysis else 0
    )
    top_category = (
        result.monthly_analysis.stable_metrics.worst_category
        if result.monthly_analysis else ""
    )

    return {
        "answer": result.feedback.feedback_message,
        "month_total": month_total,
        "top_category": str(top_category)
    }

# 3. LangSmith Dataset 생성

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "catcher-feedback-monthly-eval"

examples = [
    {
        "inputs": {"member_id": 1, "analysis_month": "2024-01"},
        "outputs": {"expected_trait": "1월 총지출과 수입 대비 비율을 언급하고, 가장 큰 지출 카테고리 구조를 분석해야 한다."}
    },
    {
        "inputs": {"member_id": 2, "analysis_month": "2024-02"},
        "outputs": {"expected_trait": "전월 대비 소비 변화를 비교하고, 줄이기 가장 쉬운 카테고리부터 개선안을 제안해야 한다."}
    },
    {
        "inputs": {"member_id": 3, "analysis_month": "2024-03"},
        "outputs": {"expected_trait": "절약 성과가 있다면 인정하고, 다음 달 구체적인 절약 목표 금액을 제시해야 한다."}
    },
    {
        "inputs": {"member_id": 4, "analysis_month": "2024-01"},
        "outputs": {"expected_trait": "구독 서비스 총 지출을 파악하고 해지 또는 조정 가능한 항목을 제안해야 한다."}
    },
    {
        "inputs": {"member_id": 1, "analysis_month": "2024-03"},
        "outputs": {"expected_trait": "3개월 누적 소비 흐름을 반영해 이번 달이 전체적으로 개선됐는지 악화됐는지 평가해야 한다."}
    },
]

existing = [d for d in client.list_datasets() if d.name == dataset_name]
if existing:
    dataset = existing[0]
    print(f"기존 dataset 사용: {dataset.name}")
else:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Catcher LLM 월간 피드백 품질 평가"
    )
    for ex in examples:
        client.create_example(
            inputs=ex["inputs"],
            outputs=ex["outputs"],
            dataset_id=dataset.id
        )
    print(f"새 dataset 생성: {dataset.name} ({len(examples)}개)")

# 4. Evaluator 정의

| Evaluator | 방식 | 월간 특화 기준 |
|---|---|---|
| `groundedness` | LLM judge | 월간 총지출·카테고리 구조가 반영됐는가 |
| `strategy_depth` | LLM judge | 다음 달 전략이 구체적이고 깊이가 있는가 |
| `actionability` | LLM judge | 즉시 실행 가능한 다음 달 목표를 제시했는가 |
| `contains_amount` | Heuristic | 구체적 금액이 포함되어 있는가 |

In [ ]:
import re
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def groundedness_evaluator(run, example):
    answer = run.outputs.get("answer", "")
    month_total = run.outputs.get("month_total", 0)
    top_category = run.outputs.get("top_category", "")

    prompt = f"""
아래 월간 피드백이 실제 소비 데이터에 근거하는지 0~1로 평가해줘.
이번 달 총지출({month_total:,}원), 최다 지출 카테고리({top_category})를 반영하면 높은 점수야.
숫자 하나만 출력해.

피드백: {answer}
"""
    score = judge_llm.invoke(prompt).content.strip()
    try:
        return {"key": "groundedness", "score": float(score)}
    except ValueError:
        return {"key": "groundedness", "score": 0.0}


def strategy_depth_evaluator(run, example):
    """다음 달 전략이 표면적이지 않고 구체적인가 (0~1)"""
    answer = run.outputs.get("answer", "")

    prompt = f"""
아래 월간 피드백의 다음 달 전략 제안이 얼마나 구체적이고 깊이 있는지 0~1로 평가해줘.
"절약하세요" 수준은 낮게, 카테고리·목표 금액·실천 방법이 명확하면 높게.
숫자 하나만 출력해.

피드백: {answer}
"""
    score = judge_llm.invoke(prompt).content.strip()
    try:
        return {"key": "strategy_depth", "score": float(score)}
    except ValueError:
        return {"key": "strategy_depth", "score": 0.0}


def actionability_evaluator(run, example):
    answer = run.outputs.get("answer", "")
    expected = example.outputs.get("expected_trait", "")

    prompt = f"""
아래 월간 피드백이 다음 달 바로 실천할 수 있는 목표나 행동을 제시하는지 0~1로 평가해줘.

기대 특성: {expected}
피드백: {answer}

숫자 하나만 출력해.
"""
    score = judge_llm.invoke(prompt).content.strip()
    try:
        return {"key": "actionability", "score": float(score)}
    except ValueError:
        return {"key": "actionability", "score": 0.0}


def contains_amount_evaluator(run, example):
    answer = run.outputs.get("answer", "")
    has_amount = bool(re.search(r'\d[\d,]*\s*(원|만원|만\s*원)', answer))
    return {"key": "contains_amount", "score": 1 if has_amount else 0}


print("evaluator 4개 정의 완료")

# 5. evaluate() 실행 → LangSmith 반영

In [ ]:
from langsmith.evaluation import evaluate

evaluate(
    target,
    data=dataset_name,
    evaluators=[
        groundedness_evaluator,
        strategy_depth_evaluator,
        actionability_evaluator,
        contains_amount_evaluator,
    ],
    experiment_prefix="monthly-feedback-v1"
)